In [2]:
from matplotlib.pyplot import *
%matplotlib inline
import os
import glob
from math import *
import pandas as pd
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable

import numpy as np
import seaborn as sns

from astropy.io import fits
from astropy.table import Table, vstack

import torch
import torch.nn.functional as F



import astropy.units as u
from astropy.coordinates import SkyCoord

In [3]:



data_path = 'data/'

  

In [4]:
files = glob.glob(data_path+"*_xSyn.fits", recursive = True)

files.sort()
run_list = []
for file in files:
    
    run_list.append(file.split("/")[-1].split("_")[0])
        
print (run_list)

['25Am02']


In [5]:
run_list_new = run_list[-1:]
print (run_list_new)


['25Am02']


In [7]:
# The GC file is obtained from Vaseliev et al. 2020
gc = Table.read(data_path+"GC_EDR3.fits")


In [8]:
err1 = 0.1
for i in range(len(run_list_new)):
    
    run = run_list_new[i]
   
    s = Table.read(data_path+"%s_xSyn.fits"%run)
    ind_sel0 = (s['CaHK_syn'] >= 0) * (s['CaHK_uncalib'] >= 15) * (s['d_CaHK'] > 0) * (s['CaHK_uncalib'] <= 31) 

    t = s[ind_sel0]
    ind_err1 = t['d_CaHK_syn'] < err1
    ind_f2 = t['flag'] == -1

    catalog = SkyCoord(ra=np.array(t['ra_x'])*u.degree, dec=np.array(t['dec_x'])*u.degree)
    ngb_id = np.array([0])
    
    for j in range(len(gc)):
    
        c = SkyCoord(gc["ra"][j]*u.deg, gc["dec"][j]*u.deg, frame='icrs')
        sep = c.separation(catalog)
        ind_ngb = sep <=  gc["rmax"][j]*u.arcmin
    
        if len(sep[ind_ngb]) > 0:

            ngb_id = np.append(ngb_id, np.array(t['source_id'][ind_ngb]))
            
    if len(ngb_id) > 1:
        
        ngb_id = ngb_id[1:]
        
    ind_ngb = np.in1d(t['source_id'], ngb_id)
        
        
    num_star_run = 0
    image_nb_run = np.unique(t['image_nb'])
    id_t = np.arange(0, len(t), 1)
    id_s = np.array(0)

    
    for k in range(len(image_nb_run)):
        
        ind_star_image = np.in1d(t['image_nb'], image_nb_run[k])
           
        delta_z = (t['CaHK_uncalib']-t['CaHK_syn'])[ind_star_image*ind_err1*~ind_ngb*ind_f2]
        delta_z_low = np.percentile(delta_z, 50) - .2
        delta_z_upp = np.percentile(delta_z, 50) + .2

            
        ind_z = (delta_z >= delta_z_low)*(delta_z <= delta_z_upp)

        id_s = np.append(id_s, id_t[ind_star_image*ind_err1*~ind_ngb*ind_f2][ind_z])

    
                      
    id_s = id_s[1:]
        
    ind_calib = np.in1d(id_t, id_s)
    t0 = t[ind_calib]
    
    x=np.array(t0['Xg']/19000)
    y=np.array(t0['Yg']/19000)

    z = np.array(t0['CaHK_uncalib'])
    z_err = np.array(t0['d_CaHK'])
    
    z0 = np.array(t0['CaHK_syn'])
    z0_err = np.array(t0['d_CaHK_syn'])

    # z_old = np.array(t0['CaHK_old'])

    f_nb = np.array(t0['image_nb'])

    ra = np.float64(t0['RA'])
    dec = np.float64(t0['Dec'])
    

    print (i, run, len(t0))

    inputs = np.vstack((x,y,z,z_err,z0,z0_err,f_nb, ra, dec)).T
    np.save(data_path+"inputs_%s.npy"%run, inputs)

    # index_run += 1
    
    # break

0 25Am02 5610


/Users/yuan/Library/Python/3.9/lib/python/site-packages/numpy/lib/function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedColumn.
  arr.partition(
